In [0]:
--Inspect the table
SELECT * FROM workspace.fintech_fraud_db.paysim_raw
LIMIT 10;

step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
500,CASH_OUT,37644.46,C1095488083,153620.0,115975.54,C133700066,0.0,37644.46,0,0
500,CASH_OUT,174109.44,C720469896,115975.54,0.0,C1043970373,1471221.71,1645331.15,0,0
500,TRANSFER,91156.75,C2096167907,100945.0,9788.25,C772269814,0.0,91156.75,0,0
500,PAYMENT,3577.83,C1450845331,9788.25,6210.42,M1237105989,0.0,0.0,0,0
500,CASH_OUT,285731.3,C1754302149,9395.0,0.0,C631141908,469326.67,755057.98,0,0
500,CASH_IN,182108.65,C331630551,100114.0,282222.65,C1696461082,311609.97,129501.32,0,0
500,CASH_IN,118899.34,C337487744,2294.0,121193.34,C1270158265,808641.5,689742.16,0,0
500,CASH_OUT,9952.33,C954693331,225070.0,215117.67,C1383116869,8823.19,18775.52,0,0
500,CASH_OUT,158469.5,C1420655012,61297.0,0.0,C596419427,2690503.82,2848973.32,0,0
500,PAYMENT,11541.34,C1701209625,17012.0,5470.66,M473889938,0.0,0.0,0,0


In [0]:
--Inspect the table structure and metadata
DESCRIBE workspace.fintech_fraud_db.paysim_raw;

col_name,data_type,comment
step,bigint,null
type,string,null
amount,double,null
nameOrig,string,null
oldbalanceOrg,double,null
newbalanceOrig,double,null
nameDest,string,null
oldbalanceDest,double,null
newbalanceDest,double,null
isFraud,bigint,null


In [0]:
--Look at table size
SELECT COUNT(*) AS total_records
FROM workspace.fintech_fraud_db.paysim_raw;

total_records
6362620


In [0]:
--Understanding transaction type and activity
SELECT 
    type, 
    COUNT(*) AS transaction_total,
    SUM(COUNT(*)) OVER () AS total_of_transaction,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () ,2) AS transaction_percentage
FROM workspace.fintech_fraud_db.paysim_raw
GROUP BY type
ORDER BY transaction_total DESC;


type,transaction_total,total_of_transaction,transaction_percentage
CASH_OUT,2237500,6362620,35.17
PAYMENT,2151495,6362620,33.81
CASH_IN,1399284,6362620,21.99
TRANSFER,532909,6362620,8.38
DEBIT,41432,6362620,0.65


In [0]:
--How many fraudulent and legitimate transactions exist within each transaction category?

SELECT
  type,
  CASE WHEN isFraud = 1 THEN 'Fraud' ELSE 'Legitimate' END AS transaction_type,
  COUNT(*) AS transaction_total
FROM workspace.fintech_fraud_db.paysim_raw
GROUP BY type, transaction_type
ORDER BY transaction_type;

type,transaction_type,transaction_total
CASH_OUT,Fraud,4116
TRANSFER,Fraud,4097
CASH_OUT,Legitimate,2233384
PAYMENT,Legitimate,2151495
DEBIT,Legitimate,41432
CASH_IN,Legitimate,1399284
TRANSFER,Legitimate,528812


In [0]:
--How many transactions are classified as fraud and how many are legigimate?
SELECT
COUNT(
    CASE
      WHEN isFraud = 1 THEN 1
      ELSE NULL
    END
  ) AS fraud,
  COUNT(
    CASE
      WHEN isFraud = 0 THEN 1
      ELSE NULL
    END
  ) AS legitimate
FROM workspace.fintech_fraud_db.paysim_raw;



fraud,legitimate
8213,6354407


In [0]:
--Which transaction types actually contain fraudulent transactions, and how many fraudulent transactions occur within each type?

SELECT
    type,
    COUNT(*) AS fraudulent_transactions
FROM workspace.fintech_fraud_db.paysim_raw
WHERE isFraud = 1
GROUP BY type
ORDER BY fraudulent_transactions;

type,fraudulent_transactions
TRANSFER,4097
CASH_OUT,4116


In [0]:
--What percentage of transactions within each category are fraudulent?
SELECT
    type,
    ROUND(COUNT(CASE WHEN isFraud = 1 THEN 1 END) * 100.0 / COUNT(*), 2) AS fraud_percentage
FROM workspace.fintech_fraud_db.paysim_raw
GROUP BY type
ORDER BY type, fraud_percentage;

type,fraud_percentage
CASH_IN,0.00
CASH_OUT,0.18
DEBIT,0.00
PAYMENT,0.00
TRANSFER,0.77


In [0]:
--What percentage of all fraudulent transactions comes from TRANSFER, and what percentage comes from CASH_OUT?
SELECT 
    type,
    COUNT(*),
    ROUND(COUNT(CASE WHEN isFraud = 1 THEN 1 END) * 100.0 / COUNT(*) FILTER (WHERE isFraud = 1), 2) AS fraud_percentage
FROM workspace.fintech_fraud_db.paysim_raw
GROUP BY type;

    

type,transaction_total,fraud_percentage
CASH_OUT,2237500,50.12
DEBIT,41432,0.00
CASH_IN,1399284,0.00
TRANSFER,532909,49.88
PAYMENT,2151495,0.00


In [0]:
-- Where the amounts involved in those fraudulent transactions differ from legitimate transactions?

SELECT
  type,
  CASE WHEN isFraud = 1 THEN 'Fraud' ELSE 'Legitimate' END AS transaction_type,
  ROUND(AVG(amount),2) AS avg_amount
FROM workspace.fintech_fraud_db.paysim_raw
GROUP BY type, transaction_type
ORDER BY type, transaction_type;

type,transaction_type,avg_amount
CASH_IN,Legitimate,168920.24
CASH_OUT,Fraud,1455102.59
CASH_OUT,Legitimate,173917.16
DEBIT,Legitimate,5483.67
PAYMENT,Legitimate,13057.6
TRANSFER,Fraud,1480891.67
TRANSFER,Legitimate,906229.01


In [0]:
--What is the minimum and maximum transaction amount among fraudulent transactions in each category?
SELECT
    type,
    MAX(CASE WHEN isFraud = 1 THEN amount END) AS Max_Amount,
    MIN(CASE WHEN isFraud = 1 THEN amount END) AS Min_Amount
FROM workspace.fintech_fraud_db.paysim_raw
GROUP BY type
ORDER BY type;

type,Max_Amount,Min_Amount
CASH_IN,null,null
CASH_OUT,1.0E7,0.0
DEBIT,null,null
PAYMENT,null,null
TRANSFER,1.0E7,63.8


In [0]:

--What is the minimum and maximum transaction amount among Legitimate transactions in each category?
SELECT
    type,
    MAX(CASE WHEN isFraud = 0 THEN amount END) AS Max_Amount,
    MIN(CASE WHEN isFraud = 0 THEN amount END) AS Min_Amount
FROM workspace.fintech_fraud_db.paysim_raw
GROUP BY type
ORDER BY type;

type,Max_Amount,Min_Amount
CASH_IN,1915267.9,0.04
CASH_OUT,2847566.62,0.01
DEBIT,569077.51,0.55
PAYMENT,238637.98,0.02
TRANSFER,9.244551664E7,2.6
